# Giga Meter — Fleet Profile

How each country's fleet actually measures: rhythm, seasonality, silence and churn,
computed **server-side in Trino** so it scales across the whole fleet. Only small
aggregates come back — a profile row, twelve seasonal rows and a spell histogram
per country — never the row-level measurements.

**What it answers**
- What is the measurement rhythm — near-daily, weekly, sparse — and how long do schools last?
- When is each country's low season, learned from its own data rather than assumed?
- Given a school has been silent X days, how likely is it to come back? (Kaplan-Meier, censoring-corrected)
- How much churn does each fleet carry, expressed as incidence per 100 school-years so countries are comparable?

**Design notes**
- Every metric is **exposure-corrected**: seasonal averages divide by school-months *in tenure*, not by calendar months present, and silence spells are treated as survival data with censoring at the last date in the data.
- Recency is measured against each country's own last data date, so a stale pipeline in one country does not look like churn.
- **Reactivation campaigns are registered explicitly** (see the CAMPAIGNS cell) — Mongolia ran one Mar–May 2026 and Sri Lanka is running one now, so their return rates and churn describe a fleet under intervention.
- The low season comes from the **published school calendar**, confirmed against the measurement dip; a purely data-derived rule produces false positives in countries with one extreme peak month.

---
## Part 0 — Setup

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3,
                         paired_shift_test, two_group_shift_test,
                         format_shift_result, bootstrap_ci,
                         wilson_ci, fmt_pct_ci,
                         kruskal_omnibus, pairwise_shift_tests)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")


In [ ]:
# =============================================================================
# SCOPE — which countries, and where to read from
# =============================================================================
COUNTRIES = []                        # ISO3 codes; [] = every country in the table
SOURCE_FILTER = "GigaMeter"           # rt_source; None = all sources
USE_TRINO = True                      # False = build from local country parquet caches
MIN_SCHOOLS = 20                      # skip countries with fewer measuring schools

CACHE_ROOT = Path("./cache")
LOW_SEASON_FRAC = 0.75                # month is "low season" below this share of the MEDIAN month
MIN_SPAN_MONTHS_FOR_SEASON = 24       # below this, seasonality is provisional (one cycle or less)

EXCLUDE_COUNTRIES = ['KAZ']           # kept out of fleet aggregates (deployment not comparable)

_iso_sql = "" if not COUNTRIES else "AND iso3_code IN ('" + "', '".join(COUNTRIES) + "')"
if EXCLUDE_COUNTRIES:
    _iso_sql += " AND iso3_code NOT IN ('" + "', '".join(EXCLUDE_COUNTRIES) + "')"
_src_sql = "" if not SOURCE_FILTER else f"AND rt_source = '{SOURCE_FILTER}'"
_scope_sql = f"{_iso_sql} {_src_sql}"
print(f"scope: {COUNTRIES or 'ALL countries'} minus {EXCLUDE_COUNTRIES or 'none'} · "
      f"source={SOURCE_FILTER or 'all'} · "
      f"{'Trino' if USE_TRINO else 'local caches'}")

In [ ]:
# =============================================================================
# CAMPAIGN REGISTRY — outreach that moves the numbers, recorded explicitly
# =============================================================================
# Reactivation campaigns manufacture returns. Any country with one inside the
# observation window will show inflated return probabilities and deflated churn
# for that period, and the effect is NOT seasonality — so it has to be recorded
# and controlled for, not absorbed into the baseline.
#
# Add entries as campaigns happen: ISO3 -> list of (start, end or None, label).
CAMPAIGNS = {
    'MNG': [('2026-03-01', '2026-05-31', 'reactivation campaign')],
    'LKA': [('2026-07-01', None,         'reactivation campaign (ongoing)')],
}

def campaign_windows(iso3):
    """[(start_ts, end_ts_or_None, label)] for a country."""
    return [(pd.Timestamp(a), pd.Timestamp(b) if b else None, lbl)
            for a, b, lbl in CAMPAIGNS.get(iso3, [])]

def in_campaign(iso3, when):
    for _a, _b, _ in campaign_windows(iso3):
        if when >= _a and (_b is None or when <= _b):
            return True
    return False

_reg = pd.DataFrame([{'iso3_code': k, 'window': f"{a} → {b or 'ongoing'}", 'label': lbl}
                     for k, v in CAMPAIGNS.items() for a, b, lbl in v])
print("Campaigns on record (edit CAMPAIGNS above as new ones run):")
display(_reg)
print("Countries in this run WITH a campaign in-window: "
      f"{[i for i in COUNTRIES if i in CAMPAIGNS] or 'none'}")

---
## Part 1 — Server-side aggregation

Four small result sets per country. The row-level table is touched once inside the
CTE chain; everything returned is already aggregated.

In [ ]:
# =============================================================================
# SQL — the shared CTE chain (school x day, gaps, tenure) reused by every query
# =============================================================================
# Audited Aug 2026. Choices that matter, stated rather than implied:
#  · the day key is the LOCAL date (`local_created_timestamp`), not the UTC `date`
#    column — they disagree on 4.3% of rows, so every query must use the same one
#  · `ref_d` is each country's own last data date, so a lagging pipeline in one
#    country does not read as churn (verified: no future-dated rows in this table)
#  · SOURCE_FILTER='GigaMeter' excludes the Mlab feed (543k rows, 2,541 schools,
#    all but 39 of which also report via Giga Meter) — school-side app only
#  · seasonality exposure runs first→last measurement, so a school contributes no
#    months after it stops: the seasonal curve describes ACTIVE schools' rhythm
#    and cannot be read as churn
BASE_CTE = f"""
WITH sd AS (                      -- one row per school-day, the only heavy step
    SELECT iso3_code, school_id_giga, CAST(local_created_timestamp AS date) AS d
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_iso_sql} {_src_sql}
    GROUP BY 1, 2, 3
),
ref AS (SELECT iso3_code, max(d) AS ref_d, min(d) AS first_d FROM sd GROUP BY 1),
lagged AS (
    SELECT iso3_code, school_id_giga, d,
           LAG(d) OVER (PARTITION BY iso3_code, school_id_giga ORDER BY d) AS prev_d
    FROM sd
),
gaps AS (
    SELECT iso3_code, school_id_giga, date_diff('day', prev_d, d) AS dur
    FROM lagged WHERE prev_d IS NOT NULL
),
ten AS (
    SELECT iso3_code, school_id_giga, min(d) AS first_d, max(d) AS last_d,
           count(*) AS days_measured,
           date_diff('day', min(d), max(d)) AS tenure_days
    FROM sd GROUP BY 1, 2
)
"""

# 1) one profile row per country
Q_PROFILE = BASE_CTE + """
, per_school AS (
    SELECT t.iso3_code, t.school_id_giga, t.days_measured, t.tenure_days,
           date_diff('day', t.last_d, r.ref_d) AS days_silent,
           t.days_measured / GREATEST(t.tenure_days / 30.44, 1.0) AS days_per_month
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
)
SELECT p.iso3_code,
       count(*) AS schools,
       approx_percentile(p.tenure_days, 0.5) / 30.44 AS median_tenure_months,
       approx_percentile(p.days_per_month, 0.5) AS median_days_per_month,
       approx_percentile(p.days_per_month, 0.25) AS p25_days_per_month,
       approx_percentile(p.days_per_month, 0.75) AS p75_days_per_month,
       sum(p.days_measured) AS school_days,
       sum(date_diff('day', t2.first_d, r.ref_d)) / 365.25 AS school_years_observed,
       -- exposure in which a silence of N days COULD have been observed: school-time
       -- beginning at least N days before the last data date (one column per horizon)
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -   7, 0)) / 365.25 AS school_years_at_risk_7,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  14, 0)) / 365.25 AS school_years_at_risk_14,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  30, 0)) / 365.25 AS school_years_at_risk_30,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  90, 0)) / 365.25 AS school_years_at_risk_90,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) - 182, 0)) / 365.25 AS school_years_at_risk_182,
       -- % silent must divide by schools that COULD be seen silent that long:
       -- a school first measuring 40 days ago cannot be 90 days silent, and
       -- counting it in the denominator flatters young fleets
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 30)  AS n_eligible_30d,
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 90)  AS n_eligible_90d,
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 182) AS n_eligible_182d,
       100.0 * count_if(p.days_silent > 30)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 30), 0)  AS pct_silent_30d,
       100.0 * count_if(p.days_silent > 90)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 90), 0)  AS pct_silent_90d,
       100.0 * count_if(p.days_silent > 182)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 182), 0) AS pct_silent_182d,
       min(r.first_d) AS data_from, min(r.ref_d) AS data_to
FROM per_school p
JOIN ten t2 ON t2.iso3_code = p.iso3_code AND t2.school_id_giga = p.school_id_giga
JOIN ref r  ON r.iso3_code = p.iso3_code
GROUP BY p.iso3_code
"""

# 2) gap-shape + weekday rhythm per country
Q_RHYTHM = BASE_CTE + """
-- day_of_week(): 1 = Monday ... 7 = Sunday, so <= 5 is Mon-Fri
, wk AS (
    SELECT iso3_code, 100.0 * count_if(day_of_week(d) <= 5) / count(*) AS pct_weekday
    FROM sd GROUP BY 1
)
SELECT g.iso3_code,
       100.0 * count_if(g.dur = 1) / count(*)              AS pct_gap_1d,
       100.0 * count_if(g.dur BETWEEN 2 AND 7) / count(*)  AS pct_gap_2_7d,
       100.0 * count_if(g.dur BETWEEN 8 AND 30) / count(*) AS pct_gap_8_30d,
       100.0 * count_if(g.dur > 30) / count(*)             AS pct_gap_over_30d,
       max(w.pct_weekday)                                  AS pct_weekday
FROM gaps g JOIN wk w ON w.iso3_code = g.iso3_code
GROUP BY g.iso3_code
"""

# 3) silence-spell histogram (completed + censored) — the input to Kaplan-Meier
Q_SPELLS = BASE_CTE + """
SELECT iso3_code, dur, 1 AS returned, count(*) AS n
FROM gaps WHERE dur >= 2 GROUP BY 1, 2
UNION ALL
SELECT t.iso3_code, date_diff('day', t.last_d, r.ref_d) AS dur, 0 AS returned, count(*) AS n
FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
WHERE date_diff('day', t.last_d, r.ref_d) >= 2
GROUP BY 1, 2
"""

# 4) exposure-corrected seasonality: mean measuring days per school-month IN TENURE
Q_SEASON = BASE_CTE + """
, spine AS (
    SELECT t.iso3_code, t.school_id_giga, m
    FROM ten t
    CROSS JOIN UNNEST(sequence(date_trunc('month', t.first_d),
                               date_trunc('month', t.last_d),
                               INTERVAL '1' MONTH)) AS x(m)
),
per_month AS (
    SELECT iso3_code, school_id_giga, date_trunc('month', d) AS m, count(*) AS days
    FROM sd GROUP BY 1, 2, 3
)
SELECT s.iso3_code, month(s.m) AS calendar_month,
       avg(COALESCE(p.days, 0)) AS mean_days_per_school_month,
       count(*) AS school_months
FROM spine s
LEFT JOIN per_month p
       ON p.iso3_code = s.iso3_code AND p.school_id_giga = s.school_id_giga AND p.m = s.m
GROUP BY 1, 2
"""
# 5) SCHOOL-level view: each school's FIRST pause of >= 2 days, so a habitual
#    pauser counts once rather than dozens of times. Same (dur, returned) shape
#    as Q_SPELLS, which lets the two be compared side by side.
Q_FIRST_SPELL = BASE_CTE + """
, spells AS (
    SELECT iso3_code, school_id_giga, d AS spell_end, date_diff('day', prev_d, d) AS dur, 1 AS returned
    FROM lagged WHERE prev_d IS NOT NULL AND date_diff('day', prev_d, d) >= 2
    UNION ALL
    SELECT t.iso3_code, t.school_id_giga, NULL, date_diff('day', t.last_d, r.ref_d), 0
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
    WHERE date_diff('day', t.last_d, r.ref_d) >= 2
),
ranked AS (
    SELECT iso3_code, school_id_giga, dur, returned,
           ROW_NUMBER() OVER (PARTITION BY iso3_code, school_id_giga
                              ORDER BY COALESCE(spell_end, DATE '9999-12-31')) AS rn
    FROM spells
)
SELECT iso3_code, dur, returned, count(*) AS n
FROM ranked WHERE rn = 1 GROUP BY 1, 2, 3
"""

print("5 queries built · the row-level table is scanned once per query, results are aggregates")

In [ ]:
# =============================================================================
# RUN — Trino if available, else rebuild the same aggregates from local caches
# =============================================================================
def _from_trino():
    cur = get_trino_cursor()
    if cur is None:
        raise RuntimeError("no Trino cursor")
    def q(sql):
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    return q(Q_PROFILE), q(Q_RHYTHM), q(Q_SPELLS), q(Q_SEASON), q(Q_FIRST_SPELL)

def _from_cache():
    """Same aggregates from <Country>/*_measurements.parquet — for offline work."""
    prof, rhy, spl, sea = [], [], [], []
    for _dir in sorted(CACHE_ROOT.iterdir()):
        if not _dir.is_dir():
            continue
        _f = sorted(_dir.glob('*_measurements.parquet'))
        if not _f:
            continue
        _iso = _dir.name[:3].upper()
        if COUNTRIES and not any(_dir.name.lower().startswith(c.lower()[:3]) or _iso == c
                                 for c in COUNTRIES):
            _match = [c for c in COUNTRIES if c in COUNTRY_ALIASES.get(_dir.name, [_iso])]
            if not _match:
                continue
            _iso = _match[0]
        _m = pd.read_parquet(_f[0], columns=['school_id_giga', 'date'])
        _m['d'] = pd.to_datetime(_m['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
        _sd = _m.dropna(subset=['d']).drop_duplicates(['school_id_giga', 'd']).sort_values(['school_id_giga', 'd'])
        _ref = _sd['d'].max()
        _sd['gap'] = _sd.groupby('school_id_giga')['d'].diff().dt.days
        _t = _sd.groupby('school_id_giga')['d'].agg(first_d='min', last_d='max', days_measured='size')
        _t['tenure_days'] = (_t['last_d'] - _t['first_d']).dt.days
        _t['days_silent'] = (_ref - _t['last_d']).dt.days
        _t['days_per_month'] = _t['days_measured'] / (_t['tenure_days'] / 30.44).clip(lower=1)
        prof.append({'iso3_code': _iso, 'schools': len(_t),
                     'median_tenure_months': _t['tenure_days'].median() / 30.44,
                     'median_days_per_month': _t['days_per_month'].median(),
                     'p25_days_per_month': _t['days_per_month'].quantile(.25),
                     'p75_days_per_month': _t['days_per_month'].quantile(.75),
                     'school_days': int(_t['days_measured'].sum()),
                     'school_years_observed': float((_ref - _t['first_d']).dt.days.sum()) / 365.25,
                     **{f'school_years_at_risk_{_h}':
                        float(((_ref - _t['first_d']).dt.days - _h).clip(lower=0).sum()) / 365.25
                        for _h in (7, 14, 30, 90, 182)},
                     'school_years_at_risk': float(((_ref - _t['first_d']).dt.days - 182)
                                                   .clip(lower=0).sum()) / 365.25,
                     **{f'n_eligible_{_h}d': int(((_ref - _t['first_d']).dt.days >= _h).sum())
                        for _h in (30, 90, 182)},
                     **{f'pct_silent_{_h}d': (100 * (_t['days_silent'] > _h).sum()
                        / max(int(((_ref - _t['first_d']).dt.days >= _h).sum()), 1))
                        for _h in (30, 90, 182)},
                     'data_from': _sd['d'].min(), 'data_to': _ref})
        _g = _sd['gap'].dropna()
        rhy.append({'iso3_code': _iso, 'pct_gap_1d': 100 * (_g == 1).mean(),
                    'pct_gap_2_7d': 100 * _g.between(2, 7).mean(),
                    'pct_gap_8_30d': 100 * _g.between(8, 30).mean(),
                    'pct_gap_over_30d': 100 * (_g > 30).mean(),
                    'pct_weekday': 100 * (_sd['d'].dt.weekday < 5).mean()})
        _sp = pd.concat([_g[_g >= 2].value_counts().rename_axis('dur').reset_index(name='n').assign(returned=1),
                         _t.loc[_t['days_silent'] >= 2, 'days_silent'].value_counts()
                           .rename_axis('dur').reset_index(name='n').assign(returned=0)])
        spl.append(_sp.assign(iso3_code=_iso))
        _exp = pd.DataFrame([(s, p) for s, (a, b) in _t[['first_d', 'last_d']].iterrows()
                             for p in pd.period_range(a.to_period('M'), b.to_period('M'), freq='M')],
                            columns=['school_id_giga', 'm'])
        _am = (_sd.assign(m=_sd['d'].dt.to_period('M')).groupby(['school_id_giga', 'm'])
               .size().rename('days').reset_index())
        _j = _exp.merge(_am, on=['school_id_giga', 'm'], how='left').fillna({'days': 0})
        _s = (_j.assign(calendar_month=_j['m'].dt.month).groupby('calendar_month')
              .agg(mean_days_per_school_month=('days', 'mean'), school_months=('days', 'size')).reset_index())
        sea.append(_s.assign(iso3_code=_iso))
    return (pd.DataFrame(prof), pd.DataFrame(rhy),
            pd.concat(spl, ignore_index=True), pd.concat(sea, ignore_index=True),
            pd.DataFrame())          # first-spell view needs Trino

COUNTRY_ALIASES = {'Fiji': ['FJI'], 'Mongolia': ['MNG'], 'Malawi': ['MWI'],
                   'South Africa': ['ZAF'], 'Uzbekistan': ['UZB'], 'Montenegro': ['MNE'],
                   'Sri Lanka': ['LKA'], 'Kenya': ['KEN'], 'Botswana': ['BWA'],
                   'Rwanda': ['RWA'], 'Zambia': ['ZMB'], 'Ethiopia': ['ETH']}

try:
    if not USE_TRINO:
        raise RuntimeError("USE_TRINO is False")
    profile, rhythm, spells, season, first_spell = _from_trino()
    _mode = "Trino (server-side)"
except Exception as _e:
    print(f"⚠️  falling back to local caches: {_e}")
    profile, rhythm, spells, season, first_spell = _from_cache()
    _mode = "local caches"

profile = profile[profile['schools'] >= MIN_SCHOOLS].set_index('iso3_code')
rhythm = rhythm.set_index('iso3_code')

In [ ]:
print(f"✓ {_mode}: {len(profile)} countries · {len(spells):,} spell rows · {len(season)} seasonal rows")

# `profile` is an intermediate frame — its analytical columns are re-presented in
# the fleet comparison table at the end. What is worth reading HERE is provenance:
# how much data each country has, and how many schools are old enough to be
# judged. Everything else stays on the frame for the calculations downstream.
coverage = pd.DataFrame({
    'schools': profile['schools'].astype(int),
    'data from': pd.to_datetime(profile['data_from']).dt.date,
    'data to': pd.to_datetime(profile['data_to']).dt.date,
    'school-years observed': profile['school_years_observed'].round(0),
    'eligible 30d': profile['n_eligible_30d'].astype(int),
    'eligible 90d': profile['n_eligible_90d'].astype(int),
    'eligible 182d': profile['n_eligible_182d'].astype(int),
    'silent >30d %': profile['pct_silent_30d'].round(0),
    'silent >90d %': profile['pct_silent_90d'].round(0),
}).sort_values('silent >30d %', ascending=False)
print("Coverage and eligibility — how much each country has been watched, and how many of its")
print("schools are old enough for each silence threshold to be observable. 'silent >30d %' uses")
print("the eligible count as its denominator, never the full school list.")
# red = more of the fleet is quiet; the two silence columns share one scale so they
# can be compared against each other as well as across countries
display(coverage.style
        .background_gradient(cmap='RdYlGn_r', subset=['silent >30d %', 'silent >90d %'], vmin=0, vmax=100)
        .format({'school-years observed': '{:,.0f}',
                 'silent >30d %': '{:.0f}%', 'silent >90d %': '{:.0f}%'}))

---
## Part 2 — Seasonality, learned per country

Months are compared with the country's **median** month. A peak-relative rule
mislabels most of the year wherever one month towers over the rest.

In [ ]:
# =============================================================================
# LOW SEASON — published school calendars as the prior, data as confirmation
# =============================================================================
# Inferring the break purely from the data produces false positives: a country
# with one towering peak month puts ordinary months below any relative line
# (Mongolia flagged Oct/Nov, Sri Lanka Nov). So the published school calendar is
# theprior, and the measurement dip either confirms it or flags a mismatch.
#
# Sources (checked Aug 2026 — update as ministries publish):
#   FJI  fiji.gov.fj / Ministry of Education: 7-week break 7 Dec – 22 Jan,
#        2-week breaks early May and late Aug–early Sep
#   MNG  school year from 1 Sep; summer vacation Jun–Aug, winter break ~Jan,
#        additional breaks Nov and late Mar–Apr
#   MWI  2025/26 calendar runs 22 Sep – 24 Jul → long break Aug–mid Sep
#   LKA  three terms Jan–Apr, Apr–Aug, Sep–Dec; breaks Dec–Jan, Apr, Aug
#   ZAF  mid-Jan to early Dec; long break Dec–early Jan, plus Mar/Apr, Jun/Jul, Sep/Oct
#   KAZ  2 Sep – 25 May; summer Jun–Aug; winter break mid-Dec – early Jan
#   UZB/MDA  Sep–May/Jun with a 13–14 week summer (among the longest in Europe)
#   ALB/BIH/MNE  Sep–Jun; summer Jul–Aug; winter break Jan
#   KEN  three terms, extended end-of-year break from late Oct/Nov into Jan
#   RWA  terms Sep–Dec, Jan–Apr, Apr–Jun; long break Jul–Aug
#   BWA/NAM/ZMB  southern-hemisphere year; long break Dec–Jan
#   BLZ/GRD/LCA/TTO/VCT  Sep–Jun; summer Jul–Aug; Christmas Dec
#   HND  Feb–Nov; long break Dec–Jan
#   BEN  Sep–Jul; long break Aug
SCHOOL_HOLIDAY_MONTHS = {
    # --- Pacific / Asia -------------------------------------------------------
    'FJI': [12, 1, 5, 8],        # 7-wk break 7 Dec-22 Jan; 2-wk breaks May, late Aug
    'MNG': [6, 7, 8, 1, 11],     # year from 1 Sep; summer Jun-Aug, winter ~Jan, Nov break
    'LKA': [12, 1, 4, 8],        # terms Jan-Apr, Apr-Aug, Sep-Dec; breaks Dec-Jan, Apr, Aug
    # --- Central Asia / Eastern Europe (Sep-May year, long summer) ------------
    'UZB': [6, 7, 8],            # ~13-14 wk summer; year resumes mid-Sep
    'KAZ': [6, 7, 8, 1],         # 2 Sep-25 May; summer Jun-Aug; winter break late Dec-early Jan
    'MDA': [6, 7, 8],            # among the longest summers in Europe (13-14 wks)
    'ALB': [7, 8],               # Sep-Jun; summer Jul-Aug
    'BIH': [7, 8, 1],            # Sep-Jun; summer Jul-Aug; winter break Jan
    'MNE': [7, 8, 1],            # Sep-Jun; summer Jul-Aug; winter break Jan
    # --- Africa ---------------------------------------------------------------
    'ZAF': [12, 1, 4, 7, 10],    # mid-Jan to early Dec; breaks Dec-Jan, Mar/Apr, Jun/Jul, Sep/Oct
    'MWI': [8, 9],               # 2025/26 year 22 Sep-24 Jul -> long break Aug-mid Sep
    'BWA': [12, 1, 4, 8],        # southern-hemisphere year; long break Dec-Jan
    'NAM': [12, 1, 5, 8],        # southern-hemisphere year; long break Dec-Jan
    'ZMB': [12, 1, 4, 8],        # terms Jan-Apr, May-Aug, Sep-Dec; long break Dec-Jan
    'KEN': [12, 1, 4, 8],        # three terms; extended end-of-year break Nov/Dec-Jan
    'RWA': [7, 8, 12],           # terms Sep-Dec, Jan-Apr, Apr-Jun; long break Jul-Aug
    'BEN': [8, 7],               # Sep-Jul year; long break Aug (Jul from mid-month)
    # --- Caribbean / Central America (Sep-Jun year) --------------------------
    'BLZ': [7, 8, 12],           # Sep-Jun; summer Jul-Aug; Christmas break Dec
    'GRD': [7, 8, 12],
    'LCA': [7, 8, 12],
    'TTO': [7, 8, 12],
    'VCT': [7, 8, 12],
    'HND': [12, 1],              # Feb-Nov year; long break Dec-Jan
}

_MON = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
_names = lambda ms: ', '.join(_MON[m - 1] for m in sorted(ms)) or '—'

LOW_SEASON, _rows = {}, []
for _iso, _g in season.groupby('iso3_code'):
    if _iso not in profile.index:
        continue
    _s = _g.set_index('calendar_month')['mean_days_per_school_month'].reindex(range(1, 13))
    _dip = set(_s[_s < LOW_SEASON_FRAC * _s.median()].dropna().index)      # data says quiet
    _pub = set(SCHOOL_HOLIDAY_MONTHS.get(_iso, []))                        # calendar says break
    if _pub:
        _confirmed = sorted(_dip & _pub)          # break AND quiet -> trust
        _source = 'calendar ∩ data'
        LOW_SEASON[_iso] = _confirmed or sorted(_pub)
    else:
        _confirmed = sorted(_dip)
        _source = 'data only (no calendar on file)'
        LOW_SEASON[_iso] = _confirmed
    _span = (pd.Timestamp(profile.loc[_iso, 'data_to']) - pd.Timestamp(profile.loc[_iso, 'data_from'])).days / 30.44
    _rows.append({'iso3_code': _iso,
                  'published break': _names(_pub) if _pub else '—',
                  'data dip': _names(_dip),
                  'LOW SEASON used': _names(LOW_SEASON[_iso]),
                  'basis': _source,
                  'dip not in calendar': _names(_dip - _pub) if _pub else '—',
                  'break without dip': _names(_pub - _dip) if _pub else '—',
                  'trough % of median': round(100 * _s.min() / _s.median()),
                  'span (mo)': round(_span),
                  'seasonality': 'provisional (<2 yrs)' if _span < MIN_SPAN_MONTHS_FOR_SEASON else 'established'})
season_summary = pd.DataFrame(_rows).set_index('iso3_code')
display(season_summary)
print("'dip not in calendar' = quiet months the school calendar does not explain (investigate: "
      "device or\nengagement problem, not a holiday). 'break without dip' = schools kept measuring "
      "through a break\n(often boarding schools or devices left on).")

_order = list(profile.index)
_ncol = 4
_nrow = int(np.ceil(len(_order) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(4.2 * _ncol, 2.6 * _nrow), squeeze=False)
_flat = axes.ravel()
for _ax in _flat[len(_order):]:
    _ax.set_visible(False)
for _ax, _iso in zip(_flat, _order):
    _s = season[season['iso3_code'] == _iso].set_index('calendar_month')['mean_days_per_school_month'].reindex(range(1, 13))
    _pub = set(SCHOOL_HOLIDAY_MONTHS.get(_iso, []))
    _cols = [GIGA_MODERATE if c in LOW_SEASON[_iso] else
             (GIGA_PRIMARY[200] if c in _pub else GIGA_PRIMARY[600]) for c in range(1, 13)]
    _ax.bar(range(1, 13), _s.values, color=_cols, edgecolor='white')
    _ax.axhline(LOW_SEASON_FRAC * _s.median(), color=GIGA_GREY[700], ls='--', lw=1)
    _ax.set_xticks(range(1, 13))
    _ax.set_xticklabels([m[0] for m in _MON], fontsize=7)
    _ax.set_title(f'{_iso}: {_names(LOW_SEASON[_iso])}', fontsize=9)
    _ax.tick_params(axis='y', labelsize=7)
    _ax.set_ylabel('days/school-mo', fontsize=7)
fig.suptitle('Seasonal rhythm — amber = low season used · pale blue = published break with no dip', y=1.005)
plt.tight_layout(); plt.show()

---
## Part 3 — Silence and churn

Kaplan–Meier per country on the spell histogram: conditional on a school already
being silent X days, how likely is it to return? Censored spells (schools quiet
right now) stay in the risk set instead of counting as failures.

In [ ]:
# =============================================================================
# CONDITIONAL SURVIVAL + CHURN INCIDENCE, PER COUNTRY
# =============================================================================
SILENCE_POINTS = [7, 14, 21, 30, 45, 60, 90, 182]
HORIZONS = [7, 14, 30, 90]

def _km(df):
    """Kaplan-Meier survivor from a (dur, returned, n) histogram."""
    _d = df.groupby(['dur', 'returned'])['n'].sum().unstack(fill_value=0)
    _d = _d.reindex(columns=[0, 1], fill_value=0).sort_index()
    _at_risk = _d.sum(axis=1)[::-1].cumsum()[::-1]
    _S, _out = 1.0, {}
    for _t, _row in _d.iterrows():
        if _row[1] > 0:
            _S *= (1 - _row[1] / _at_risk[_t])
        _out[_t] = _S
    return pd.Series(_out)

CHURN_HORIZONS = [7, 14, 30, 90, 182]   # 'churned' at each definition of silence
# NOTE 7d and 14d are NOT churn in any real sense — 88% of pauses are <=7 days and
# ~93% of schools silent 7 days come back. They are included so the rate can be read
# as a gradient: the drop from @7d to @182d is how much silence is temporary.
MIN_AT_RISK_YEARS = 20             # below this, the rate is not reportable

surv_tables, churn_rows = {}, []
for _iso in profile.index:
    _K = _km(spells[spells['iso3_code'] == _iso])
    _surv = lambda x, _K=_K: (_K[_K.index <= x].iloc[-1] if len(_K[_K.index <= x]) else 1.0)
    _sp_i = spells[spells['iso3_code'] == _iso]
    _t = pd.DataFrame({'silent_for_days': SILENCE_POINTS})
    _t['risk set'] = [int(_sp_i.loc[_sp_i['dur'] >= X, 'n'].sum()) for X in SILENCE_POINTS]
    _t['still silent'] = [int(_sp_i.loc[(_sp_i['dur'] >= X) & (_sp_i['returned'] == 0), 'n'].sum())
                          for X in SILENCE_POINTS]
    for _h in HORIZONS:
        _t[f'+{_h}d %'] = [round(100 * (1 - _surv(X + _h) / _surv(X))) for X in SILENCE_POINTS]
    _t['by 365d % (spells)'] = [round(100 * (1 - _surv(365) / _surv(X))) for X in SILENCE_POINTS]
    # School-level companion: one spell per school (its FIRST pause), so a school
    # that pauses every December is counted once instead of dozens of times. The
    # spell view answers "of pauses this long, how many end?"; this one answers
    # "of SCHOOLS that go this quiet, how many come back?".
    if len(first_spell):
        _fs = first_spell[first_spell['iso3_code'] == _iso]
        if len(_fs):
            _Kf = _km(_fs)
            _survf = lambda x, _Kf=_Kf: (_Kf[_Kf.index <= x].iloc[-1] if len(_Kf[_Kf.index <= x]) else 1.0)
            _t['by 365d % (schools)'] = [round(100 * (1 - _survf(365) / _survf(X))) for X in SILENCE_POINTS]
            _t['schools at X'] = [int(_fs.loc[_fs['dur'] >= X, 'n'].sum()) for X in SILENCE_POINTS]
    surv_tables[_iso] = _t.set_index('silent_for_days')

    # churn incidence at each horizon. Exposure MUST match the definition: a
    # silence of N days is only observable in school-time that began >= N days
    # before the last data date — otherwise young fleets score a fake zero.
    _row = {'iso3_code': _iso,
            'schools': int(profile.loc[_iso, 'schools']),
            'schools silent now >30d': int(round(profile.loc[_iso, 'pct_silent_30d']
                                                 * profile.loc[_iso, 'n_eligible_30d'] / 100)),
            'schools silent now >90d': int(round(profile.loc[_iso, 'pct_silent_90d']
                                                 * profile.loc[_iso, 'n_eligible_90d'] / 100))}
    for _h in CHURN_HORIZONS:
        _events = int(_sp_i[(_sp_i['dur'] >= _h) & (_sp_i['returned'] == 0)]['n'].sum())
        _risk = float(profile.loc[_iso, f'school_years_at_risk_{_h}'])
        _row[f'silent>{_h}d (n)'] = _events
        _row[f'yrs at risk {_h}d'] = round(_risk)
        _row[f'churn/100 yrs @{_h}d'] = round(100 * _events / _risk, 1) if _risk >= MIN_AT_RISK_YEARS else np.nan
    _row['P(return | 30d) %'] = int(surv_tables[_iso].loc[30, 'by 365d % (spells)'])
    _row['P(return | 90d) %'] = int(surv_tables[_iso].loc[90, 'by 365d % (spells)'])
    churn_rows.append(_row)

churn = pd.DataFrame(churn_rows).set_index('iso3_code')
churn = churn.sort_values('churn/100 yrs @182d', ascending=False, na_position='last')
print("CHURN INCIDENCE at three definitions of 'gone' — events per 100 school-years AT RISK\n")
print("  silent>Nd (n)      schools whose CURRENT silence has already run N+ days (still silent")
print("                     at the last data date) — i.e. observed churn events")
print("  yrs at risk Nd     school-time in which such an event COULD have been seen: for each")
print("                     school, (days from its first measurement to the country's last data")
print("                     date) minus N, floored at zero, summed and divided by 365.25. A school")
print("                     that started 60 days ago contributes nothing to the 90-day column,")
print("                     because it cannot yet have been silent that long.")
print("  churn/100 yrs @Nd  events / at-risk years x 100 — the exposure-matched rate, comparable")
print("                     across countries and across fleet ages. Blank = too little exposure.\n")
_rate_cols = [f'churn/100 yrs @{_h}d' for _h in CHURN_HORIZONS]
_show = ['schools', 'schools silent now >30d', 'schools silent now >90d'] + \
        [c for _h in CHURN_HORIZONS for c in (f'silent>{_h}d (n)', f'churn/100 yrs @{_h}d')]
_vmax = float(np.nanpercentile(churn[_rate_cols].to_numpy(dtype=float), 95))
display(churn[_show].style
        .background_gradient(cmap='RdYlGn_r', subset=_rate_cols, vmin=0, vmax=_vmax)
        .format({c: '{:.1f}' for c in _rate_cols}, na_rep='—'))
print("(at-risk exposure columns are used in the rates and omitted here; "
      "see 'yrs at risk' in the churn frame)")
_thin = churn[churn['churn/100 yrs @182d'].isna()]
if len(_thin):
    print(f"no 182-day rate (under {MIN_AT_RISK_YEARS} school-years at risk): {', '.join(_thin.index)} — "
          f"too young for the event to be observable, which is NOT zero churn. Their 30- and 90-day "
          f"rates are still valid.")

fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(churn))))
_p = churn.dropna(subset=['churn/100 yrs @182d']).sort_values('churn/100 yrs @182d')
_y = np.arange(len(_p)); _h = 0.26
for _k, (_hz, _c) in enumerate(zip(CHURN_HORIZONS, [GIGA_PRIMARY[200], GIGA_PRIMARY[500], GIGA_PRIMARY[800]])):
    ax.barh(_y + (_k - 1) * _h, _p[f'churn/100 yrs @{_hz}d'], height=_h, color=_c, label=f'silent >{_hz}d')
ax.set_yticks(_y); ax.set_yticklabels(_p.index, fontsize=8)
ax.set_xlabel('churn events per 100 school-years at risk')
ax.set_title('Churn incidence by definition of "gone"')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for _iso, _t in surv_tables.items():
    print(f"\n{_iso} — given a school is already silent for X days, % that return."
          f"\n  risk set = silence spells that actually reached X days (completed + still running);"
          f"\n  still silent = of those, the ones ongoing at the last data date (censored, not failures)."
          f"\n  'by 365d % (spells)' counts EPISODES — a school that pauses every year contributes one"
          f"\n  each time. 'by 365d % (schools)' counts each school once, using its FIRST pause, and"
          f"\n  'schools at X' is that school count. Quote the school column to country teams."
          f"\n  Rows with a risk set under ~30 are unstable.")
    display(_t)

fig, ax = plt.subplots(figsize=(9, 4))
for _iso, _t in surv_tables.items():
    ax.plot(_t.index, _t['by 365d % (spells)'], marker='o', label=_iso)
ax.set_xlabel('days already silent'); ax.set_ylabel('% returning within a year')
ax.set_title('Silence is not the same thing in every country')
ax.legend(); plt.tight_layout(); plt.show()

---
## Part 3b — Campaign control

Returns are not always organic. Where a reactivation campaign ran, the return
spike belongs to the campaign, and the country's headline numbers describe a
fleet under intervention.

In [ ]:
# =============================================================================
# RETURN EVENTS BY MONTH — so campaign effects are visible, not baked in
# =============================================================================
Q_RETURNS_MONTHLY = BASE_CTE + """
SELECT iso3_code,
       date_trunc('month', d) AS return_month,
       count_if(dur BETWEEN 31 AND 90)  AS returns_31_90d,
       count_if(dur > 90)               AS returns_over_90d,
       count(*)                         AS returns_over_30d
FROM (
    SELECT iso3_code, d, date_diff('day', prev_d, d) AS dur
    FROM lagged WHERE prev_d IS NOT NULL
) WHERE dur > 30
GROUP BY 1, 2
"""

def _returns_from_cache():
    _out = []
    for _dir in sorted(CACHE_ROOT.iterdir()):
        _f = sorted(_dir.glob('*_measurements.parquet')) if _dir.is_dir() else []
        if not _f:
            continue
        _iso = next((c for c in COUNTRY_ALIASES.get(_dir.name, []) if not COUNTRIES or c in COUNTRIES), None)
        if _iso is None or (COUNTRIES and _iso not in COUNTRIES):
            continue
        _m = pd.read_parquet(_f[0], columns=['school_id_giga', 'date'])
        _m['d'] = pd.to_datetime(_m['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
        _sd = _m.dropna(subset=['d']).drop_duplicates(['school_id_giga', 'd']).sort_values(['school_id_giga', 'd'])
        _sd['dur'] = _sd.groupby('school_id_giga')['d'].diff().dt.days
        _r = _sd[_sd['dur'] > 30].copy()
        _r['return_month'] = _r['d'].dt.to_period('M').dt.to_timestamp()
        _g = _r.groupby('return_month').agg(
            returns_31_90d=('dur', lambda s: int(s.between(31, 90).sum())),
            returns_over_90d=('dur', lambda s: int((s > 90).sum())),
            returns_over_30d=('dur', 'size')).reset_index()
        _out.append(_g.assign(iso3_code=_iso))
    return pd.concat(_out, ignore_index=True) if _out else pd.DataFrame()

try:
    if not USE_TRINO:
        raise RuntimeError('USE_TRINO is False')
    _cur = get_trino_cursor()
    _cur.execute(Q_RETURNS_MONTHLY)
    returns_monthly = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description])
except Exception as _e:
    print(f"⚠️  returns-by-month from local caches: {_e}")
    returns_monthly = _returns_from_cache()
returns_monthly['return_month'] = pd.to_datetime(returns_monthly['return_month'])

_order = list(profile.index)
_ncol = 4
_nrow = int(np.ceil(len(_order) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(4.4 * _ncol, 2.7 * _nrow), squeeze=False)
_flat = axes.ravel()
for _ax in _flat[len(_order):]:
    _ax.set_visible(False)
for _ax, _iso in zip(_flat, _order):
    _g = returns_monthly[returns_monthly['iso3_code'] == _iso].sort_values('return_month')
    _ax.bar(_g['return_month'], _g['returns_over_30d'], width=20, color=GIGA_PRIMARY[600])
    for _a, _b, _lbl in campaign_windows(_iso):
        _ax.axvspan(_a, _b or _g['return_month'].max(), color=GIGA_MODERATE, alpha=0.30)
        _ax.text(_a, _ax.get_ylim()[1] * 0.92, ' campaign', fontsize=8, color=GIGA_GREY[800])
    for _m in LOW_SEASON.get(_iso, []):
        for _yr in _g['return_month'].dt.year.unique():
            _ax.axvline(pd.Timestamp(year=int(_yr), month=int(_m), day=15), color=GIGA_GREY[300], lw=6, alpha=.25)
    _ax.set_title(f'{_iso}', fontsize=9)
    _ax.tick_params(axis='x', rotation=45, labelsize=6)
    _ax.tick_params(axis='y', labelsize=7)
    _ax.set_ylabel('returns', fontsize=7)
fig.suptitle('Schools returning after >30d silence, by month (amber = campaign window, grey = low season)', y=1.005)
plt.tight_layout(); plt.show()

# how much of each country's return volume sits inside a campaign window
_rows = []
for _iso in profile.index:
    _g = returns_monthly[returns_monthly['iso3_code'] == _iso]
    if _g.empty:
        continue
    _inw = _g[[in_campaign(_iso, t) for t in _g['return_month']]]
    _rows.append({'iso3_code': _iso, 'returns >30d (total)': int(_g['returns_over_30d'].sum()),
                  'inside campaign window': int(_inw['returns_over_30d'].sum()),
                  '% campaign-driven': round(100 * _inw['returns_over_30d'].sum() /
                                             max(_g['returns_over_30d'].sum(), 1), 1),
                  'campaign in window': bool(CAMPAIGNS.get(_iso))})
campaign_effect = pd.DataFrame(_rows).set_index('iso3_code')
display(campaign_effect)
print("Where '% campaign-driven' is material, the country's return probabilities and churn rate")
print("describe a fleet UNDER INTERVENTION — compare it with its own pre-campaign baseline, not")
print("with countries that had no campaign.")

---
## Part 3c — Age-aligned comparison

Churn rates are not comparable between a four-year-old fleet and a
six-month-old one. Aligning on school tenure removes fleet age from the
comparison and is the control that a "provisional" flag only approximates.


In [ ]:
# =============================================================================
# TENURE-ALIGNED SURVIVAL — compare fleets at the same AGE, not the same date
# =============================================================================
# Countries started years apart, so raw churn compares a 4-year-old fleet with a
# 6-month-old one. Aligning on tenure removes that: for every school, how long
# did it keep measuring after its FIRST measurement? Each age column counts only
# schools with at least that much follow-up in their own country, so a young
# fleet contributes to the columns it has earned and no others.
AGES = [3, 6, 12, 18, 24]

_age_sql = ",\n       ".join(
    f"count_if(followup_m >= {a}) AS n_age{a}, "
    f"round(100.0 * count_if(followup_m >= {a} AND lifespan_m >= {a}) "
    f"/ NULLIF(count_if(followup_m >= {a}), 0), 0) AS alive_{a}mo" for a in AGES)
Q_TENURE = f"""
WITH sd AS (
    SELECT iso3_code, school_id_giga, CAST(local_created_timestamp AS date) AS d
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_scope_sql}
    GROUP BY 1, 2, 3
),
ten AS (SELECT iso3_code, school_id_giga, min(d) AS first_d, max(d) AS last_d FROM sd GROUP BY 1, 2),
ref AS (SELECT iso3_code, max(d) AS ref_d FROM sd GROUP BY 1),
elig AS (
    SELECT t.iso3_code, t.school_id_giga,
           date_diff('month', t.first_d, r.ref_d)  AS followup_m,
           date_diff('month', t.first_d, t.last_d) AS lifespan_m
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
)
SELECT iso3_code,
       {_age_sql}
FROM elig GROUP BY 1
HAVING count_if(followup_m >= 6) >= 30
ORDER BY alive_12mo DESC NULLS LAST
"""

try:
    _cur = get_trino_cursor()
    _cur.execute(Q_TENURE)
    tenure_survival = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description]).set_index('iso3_code')
    print("% of schools still measuring N months after their first measurement "
          "(n = schools with that much follow-up):")
    display(tenure_survival)

    _plot = tenure_survival[[f'alive_{a}mo' for a in AGES]].astype(float)
    _plot = _plot.mask(tenure_survival[[f'n_age{a}' for a in AGES]].astype(float).lt(30).values)
    _big = tenure_survival['n_age12'].astype(float).fillna(0) >= 50
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for _iso, _row in _plot[_big].iterrows():
        ax.plot(AGES, _row.values, marker='o', label=_iso)
    ax.set_xticks(AGES); ax.set_xlabel('months since first measurement'); ax.set_ylim(0, 100)
    ax.set_ylabel('% of schools still measuring')
    ax.set_title('Fleet survival at the same age (countries with >= 50 schools at 12 months)')
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()
    # FLEET total — pooled across every country, plus a fleet median of country rates
    _fleet_row = {}
    for _a in AGES:
        _n = tenure_survival[f'n_age{_a}'].astype(float)
        _alive = tenure_survival[f'alive_{_a}mo'].astype(float)
        _valid = _n.notna() & _alive.notna() & (_n > 0)
        _fleet_row[f'n_age{_a}'] = int(_n[_valid].sum())
        _fleet_row[f'alive_{_a}mo'] = round(float((_n[_valid] * _alive[_valid]).sum() / _n[_valid].sum()), 0)
    _median_row = {f'n_age{_a}': int(tenure_survival[f'n_age{_a}'].astype(float).gt(0).sum()) for _a in AGES}
    _median_row.update({f'alive_{_a}mo': round(float(
        tenure_survival.loc[tenure_survival[f'n_age{_a}'].astype(float) >= 30, f'alive_{_a}mo'].astype(float).median()), 0)
        for _a in AGES})
    survivorship = pd.concat([
        tenure_survival,
        pd.DataFrame([_fleet_row, _median_row], index=['FLEET (pooled schools)', 'FLEET (median country, n>=30)'])])
    print("\nSURVIVORSHIP — % of schools still measuring at 3 / 6 / 12 / 18 / 24 months")
    display(survivorship[[c for _a in AGES for c in (f'n_age{_a}', f'alive_{_a}mo')]])

    _exp = CACHE_ROOT / '_fleet'
    _exp.mkdir(exist_ok=True)
    survivorship.to_csv(_exp / 'fleet_survivorship.csv')
    print(f"exported {_exp / 'fleet_survivorship.csv'}")
    print("Read down a column, never across a row: a country missing alive_24mo simply has not")
    print("existed that long. Small n at older ages is also a survivorship sample — the schools")
    print("with 24 months of follow-up are early adopters, who are not typical of later cohorts.")
except Exception as _e:
    tenure_survival = pd.DataFrame()
    print(f"(tenure-aligned survival needs Trino: {_e})")

---
## Part 3d — App version of the live fleet

Which still-measuring schools are on an old build. Restricted to active
schools deliberately: an abandoned school on an old version is a churn
problem, not an upgrade problem.


In [ ]:
# =============================================================================
# APP VERSION OF SCHOOLS STILL MEASURING — who is stuck on an old build?
# =============================================================================
# Version is taken from each school's MOST RECENT measurement, and only schools
# still measuring (silent <= ACTIVE_DAYS) are counted — an abandoned school on
# 1.0.4 is a churn problem, not an upgrade problem.
# 1.0.9 is the cut asked for: below it the app predates the 2024-11 line, and
# device_id (2.0.2+) and several later fields are unavailable.
ACTIVE_DAYS = 30
OLD_BELOW = '1.0.9'

Q_APPVER = f"""
WITH last_meas AS (
    -- local date everywhere, to match BASE_CTE (UTC `date` differs on 4.3% of rows)
    SELECT iso3_code, school_id_giga, app_version,
           CAST(local_created_timestamp AS date) AS d,
           ROW_NUMBER() OVER (PARTITION BY iso3_code, school_id_giga
                              ORDER BY CAST(local_created_timestamp AS date) DESC) AS rn
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_scope_sql}
),
ref AS (SELECT iso3_code, max(CAST(local_created_timestamp AS date)) AS ref_d
        FROM default.all_gigameter_measurement_data
        WHERE local_created_timestamp IS NOT NULL {_scope_sql} GROUP BY 1),
cur AS (
    SELECT l.iso3_code, l.school_id_giga, l.app_version,
           date_diff('day', l.d, r.ref_d) AS days_silent
    FROM last_meas l JOIN ref r ON r.iso3_code = l.iso3_code
    WHERE l.rn = 1
)
SELECT iso3_code,
       count_if(days_silent <= {ACTIVE_DAYS}) AS active_schools,
       count_if(days_silent <= {ACTIVE_DAYS} AND
                (app_version IS NULL OR
                 CAST(split_part(app_version,'.',1) AS INTEGER) < 1 OR
                 (CAST(split_part(app_version,'.',1) AS INTEGER) = 1 AND
                  CAST(split_part(app_version,'.',2) AS INTEGER) = 0 AND
                  CAST(split_part(app_version,'.',3) AS INTEGER) < 9))) AS active_below_1_0_9,
       count_if(days_silent <= {ACTIVE_DAYS} AND
                CAST(split_part(app_version,'.',1) AS INTEGER) >= 2) AS active_on_2x,
       count_if(days_silent <= {ACTIVE_DAYS} AND app_version = '2.0.3') AS active_on_latest
FROM cur GROUP BY 1 HAVING count_if(days_silent <= {ACTIVE_DAYS}) > 0
ORDER BY 3 DESC
"""

try:
    _cur = get_trino_cursor()
    _cur.execute(Q_APPVER)
    appver = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description]).set_index('iso3_code')
    appver['% below 1.0.9'] = (100 * appver['active_below_1_0_9'] / appver['active_schools']).round(0)
    appver['% on 2.x'] = (100 * appver['active_on_2x'] / appver['active_schools']).round(0)
    appver['% on 2.0.3'] = (100 * appver['active_on_latest'] / appver['active_schools']).round(0)
    _tot = {'active_schools': int(appver['active_schools'].sum()),
            'active_below_1_0_9': int(appver['active_below_1_0_9'].sum()),
            'active_on_2x': int(appver['active_on_2x'].sum()),
            'active_on_latest': int(appver['active_on_latest'].sum())}
    _tot['% below 1.0.9'] = round(100 * _tot['active_below_1_0_9'] / _tot['active_schools'])
    _tot['% on 2.x'] = round(100 * _tot['active_on_2x'] / _tot['active_schools'])
    _tot['% on 2.0.3'] = round(100 * _tot['active_on_latest'] / _tot['active_schools'])
    appver = pd.concat([appver, pd.DataFrame([_tot], index=['FLEET'])])
    print(f"Schools measuring in the last {ACTIVE_DAYS} days, by app version of their latest test:")
    display(appver[['active_schools', 'active_below_1_0_9', '% below 1.0.9', '% on 2.x', '% on 2.0.3']])

    _p = appver.drop(index='FLEET')
    _p = _p[_p['active_schools'] >= 20].sort_values('% below 1.0.9', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(_p))))
    ax.barh(_p.index, _p['% below 1.0.9'], color=GIGA_BAD, label=f'< {OLD_BELOW}')
    ax.barh(_p.index, _p['% on 2.x'], left=_p['% below 1.0.9'], color=GIGA_GOOD, label='2.x')
    ax.set_xlabel('% of still-measuring schools'); ax.set_xlim(0, 100)
    ax.set_title(f'App version of schools still measuring (last {ACTIVE_DAYS} days)')
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()
    print(f"FLEET: {_tot['active_below_1_0_9']:,} of {_tot['active_schools']:,} still-measuring schools "
          f"({_tot['% below 1.0.9']:.0f}%) run a build older than {OLD_BELOW}.")
    print("These are reachable — they are measuring right now — so they are the addressable "
          "upgrade\nqueue, unlike silent schools which need reactivation first.")
except Exception as _e:
    appver = pd.DataFrame()
    print(f"(app-version breakdown needs Trino: {_e})")

---
## Part 4 — Fleet comparison

One row per country: rhythm, seasonality and churn side by side. This is the table
to track over time and to hand to country teams.

In [ ]:
# =============================================================================
# FLEET PROFILE — one row per country
# =============================================================================
fleet = pd.DataFrame({
    'schools': profile['schools'].astype(int),
    'data span (mo)': ((pd.to_datetime(profile['data_to']) - pd.to_datetime(profile['data_from']))
                       .dt.days / 30.44).round(0),
    'median tenure (mo)': profile['median_tenure_months'].round(1),
    'median days/active mo': profile['median_days_per_month'].round(1),
    'gap = 1 day %': rhythm['pct_gap_1d'].round(0),
    'weekday %': rhythm['pct_weekday'].round(0),
    # index-aligned Series everywhere — a positional list here silently mismatches
    # rows when the frames carry differently ordered indexes
    'low season': pd.Series({i: ', '.join(_MON[c - 1] for c in LOW_SEASON.get(i, [])) or '—'
                             for i in profile.index}),
    'trough % of median': season_summary['trough % of median'].reindex(profile.index),
    'silent >90d now %': profile['pct_silent_90d'].where(profile['n_eligible_90d'] >= 20).round(0),
    'n eligible 90d': profile['n_eligible_90d'].astype(int),
    'P(return | 90d) % (spells)': churn['P(return | 90d) %'],
    'churn /100 yrs @90d': churn['churn/100 yrs @90d'],
    'churn /100 yrs @182d': churn['churn/100 yrs @182d'],
    'campaign in window': pd.Series({i: ('yes: ' + '; '.join(l for _, _, l in campaign_windows(i)))
                                     if CAMPAIGNS.get(i) else 'no' for i in profile.index}),
})
fleet = fleet.reindex(profile.index)      # keep one canonical row order
_rhythm_label = np.select(
    [fleet['median days/active mo'] >= 15, fleet['median days/active mo'] >= 8],
    ['near-daily', 'several days/week'], default='sparse')
fleet.insert(4, 'rhythm', _rhythm_label)
display(fleet)

_out = CACHE_ROOT / '_fleet'
_out.mkdir(exist_ok=True)
fleet.to_csv(_out / 'fleet_profile.csv')
season.to_csv(_out / 'fleet_seasonality.csv', index=False)
pd.concat({k: v for k, v in surv_tables.items()}, names=['iso3_code']).to_csv(_out / 'fleet_survival.csv')
print(f"exported to {_out}/ (fleet_profile, fleet_seasonality, fleet_survival)")
print("\nReading guide: 'rhythm' and 'low season' describe how a fleet behaves; "
      "'P(return | 90d)' and\n'churn /100 school-yrs' describe how much of it is being lost — "
      "the second pair is only\ncomparable across countries because both are exposure-corrected.")